# **ReliefWeb API to Fetch Disaster Reports**
- Fetches disaster reports (Flood, Earthquake, Conflict) from ReliefWeb and stores them in a CSV file

**You need an approved appname to use this script**
- Go to ReliefWeb API docs (https://apidoc.reliefweb.int/parameters#appname)
- Fill in the short request form with your name, email, and use case
- Wait for approval email with your appname
- Paste your approved appname in the `APPNAME` field below

**Run In Google Colab**

**ReliefWeb API Docs**
[API Documentation](https://apidoc.reliefweb.int/)

In [ ]:
!pip install requests pandas

In [ ]:
import requests
import pandas as pd
import time

In [ ]:
def fetch_reliefweb_data(disaster_type, appname, limit=500):
    """
    Fetches disaster reports from the ReliefWeb API v2.
    Paginates automatically until `limit` records are collected.

    Args:
        disaster_type (str): Keyword to search e.g. 'Flood', 'Earthquake', 'Conflict'
        appname (str):       Your approved ReliefWeb appname
        limit (int):         Max number of records to fetch (default: 500)

    Returns:
        list of dicts with keys: text, label, date, country, source
    """
    url = "https://api.reliefweb.int/v2/reports"
    params = {"appname": appname}

    records = []
    offset = 0
    batch_size = 100  # max safe batch size for ReliefWeb API

    print(f"Fetching reports for: {disaster_type}...")

    while len(records) < limit:
        payload = {
            "query": {
                "value": disaster_type,
                "fields": ["title", "body"],
                "operator": "AND"
            },
            "fields": {
                "include": ["title", "body", "date.created", "country.name", "source.name"]
            },
            "filter": {
                "field": "status",
                "value": ["published"]
            },
            "sort": ["date.created:desc"],
            "limit": batch_size,
            "offset": offset
        }

        try:
            response = requests.post(url, params=params, json=payload, timeout=30)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.HTTPError as e:
            print(f"  HTTP Error: {e}")
            print(f"  API message: {response.text[:300]}")
            break
        except requests.exceptions.RequestException as e:
            print(f"  Request failed: {e}")
            break

        items = data.get("data", [])
        total_available = data.get("totalCount", 0)

        if not items:
            print(f"  No more results found.")
            break

        for item in items:
            fields = item.get("fields", {})
            title = fields.get("title", "").strip()
            body  = fields.get("body",  "").strip()
            combined = (title + " " + body).strip()

            if len(combined) > 100:
                country_list = fields.get("country", [])
                source_list  = fields.get("source",  [])
                records.append({
                    "text":    combined,
                    "label":   disaster_type,
                    "date":    fields.get("date", {}).get("created", ""),
                    "country": ", ".join(c.get("name", "") for c in country_list),
                    "source":  ", ".join(s.get("name", "") for s in source_list)
                })

        offset += len(items)
        print(f"  Collected {len(records)} records | Total available on API: {total_available}")

        # stop if we reached the end of available results
        if offset >= total_available or len(items) < batch_size:
            break

        time.sleep(0.5)  # be polite to the API

    print(f"  Done — {len(records)} records collected for '{disaster_type}'\n")
    return records[:limit]

In [ ]:
# ── paste your approved appname here ──────────────────────────────────────
APPNAME = "YOUR APPROVED APPNAME HERE"

# fetch reports for 3 disaster categories
floods    = fetch_reliefweb_data("Flood",      appname=APPNAME, limit=500)
quakes    = fetch_reliefweb_data("Earthquake", appname=APPNAME, limit=500)
conflicts = fetch_reliefweb_data("Conflict",   appname=APPNAME, limit=500)

# combine all records into one dataframe
all_records = floods + quakes + conflicts

df = pd.DataFrame(all_records)
df.drop_duplicates(subset=["text"], inplace=True)
df.reset_index(drop=True, inplace=True)

df.to_csv("reliefweb_disaster_reports.csv", index=False)
print(f"✅ Fetched {len(df)} reports and saved to 'reliefweb_disaster_reports.csv'.")
print(df.shape)
df.head(10)